<a href="https://colab.research.google.com/github/ABBuriro/WideConvNet/blob/main/RegCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hamid M. et al. Deep Convolutional Neural Network Regularization for
Alcoholism Detection Using EEG Signals

---
For comparison with WideConvNet, suggested by one of the reviewer

---

By: Abdul Baseer Buriro  
email: abdul.baseer@iba-suk.edu.pk  
December 22, 2024

In [ ]:
# Getting tensor flow to Build a 1D CNN with two layers (same as used in WST)
from IPython.display import clear_output
import tensorflow as tf
from tensorflow import keras
from keras import models, optimizers, utils

from keras.models import Sequential
from keras.utils import get_custom_objects

# Libraries for Model Building and EEGNet
from keras.layers import Dense, Activation, Dropout, Conv1D, Input, Flatten
from keras.layers import Average, Reshape, concatenate
from keras.layers import MaxPooling1D, BatchNormalization, AveragePooling1D
from keras.layers import Conv2D, AveragePooling2D, MaxPooling2D,GlobalMaxPool1D
from keras.layers import SeparableConv2D, DepthwiseConv2D, SpatialDropout2D
from keras.layers import SeparableConv1D, DepthwiseConv1D, SpatialDropout1D
from keras.constraints import max_norm
from keras.regularizers import l1_l2
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model

# Compilation
from keras.losses import categorical_crossentropy, binary_crossentropy
from keras.optimizers import Adam, SGD, RMSprop

# Call Backs (Early stopping)
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
early_stopping = EarlyStopping(monitor= 'val_accuracy',
                               patience=5,
                               min_delta=0.001,
                               mode='max'
                               )

# Performance metrics
from keras import metrics
clear_output()

# Importing required libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score,StratifiedKFold
from scipy.io import loadmat

In [ ]:
# Subjects and their corresponding labels (i.e., alcoholic/control)
Sub = np.array(['co2a0000365', 'co2a0000368',  'co2a0000369',  'co2a0000372',
                'co2a0000375', 'co2a0000377',  'co2a0000385',  'co2a0000392',
                'co2a0000398', 'co2a0000400',  'co2a0000403',  'co2a0000404',
                'co2a0000405', 'co2a0000406',  'co2a0000407',  'co2a0000409',
                'co2a0000410', 'co2a0000414',  'co2a0000415',  'co2a0000416',
                #
                'co2c0000339',  'co2c0000340', 'co2c0000341', 'co2c0000342',
                'co2c0000344',  'co2c0000345', 'co2c0000346', 'co2c0000347',
                'co2c0000348',  'co2c0000351', 'co2c0000354', 'co2c0000356',
                'co2c0000357',  'co2c0000363', 'co2c0000374', 'co2c0000383',
                'co2c0000389', 'co2c0000393',  'co2c0000397', 'co2c1000367'
                ])
Y = np.array(['a'] * 20 + ['c'] * 20)

# Functions used in the main program
# ------------------------------------------------------------------------------
# 1 -Data concatenation. This function is to be used to concatenate the features
# and labels of corresponding train/test subjects
# 256 is the sampling frequency and 16 is the number of trials used
def Concatenate(subjects):
  L = len(subjects)
  EEG = np.zeros(shape=(1,16),dtype=float)
  Lab = np.zeros((1),dtype=int)
  for i in range(L):
    data = loadmat(folder+subjects[i]+'.mat')
    if np.isnan(data['ndata']).any():
      print(subjects[i])
      break
    EEG = np.concatenate([EEG,data['ndata'][0:256*16,0:16]])
    Lab = np.concatenate([Lab,data['ndata'][0:256*16,16]])
  EEG = np.delete(EEG,0,0)
  Lab = np.delete(Lab,0,0)
  Lab = Lab.ravel()
  return (Lab, EEG)


# Preprocessing (data formatting) to use EEGNet, DeepConvet, and ShallowConvet
# NOTE: The EEG data (FULL) on UCI is a file for each trail and 120 trails for
# each subject. Total number of files = 120 trail/sub * 122 sub = 14,640 files
# OR 14,640 seconds of EEG.
# In this study, 20 alcoholic and 20 healthy subjects are included and 16 trails
# for each of 16 channels EEG as published in earlier paper in 2021.
def Preprocessing_EEGNet(X_train, Y_tain, X_test, Y_test):

  fs = 256; # sampling frequency
  trails = 16;
  ch = 16; # Number of channels
  L_tr, L_ts = int(len(Y_train)/fs), int(len(Y_test)/fs)

  Lab_tr = np.zeros(shape=(L_tr,1))
  for i in range(L_tr):
    Lab_tr[i] = int(sum(Y_train[i*fs:(i+1)*fs])/fs)

  Lab_tr = to_categorical(Lab_tr, num_classes=2)
  data_tr = X_train.reshape(L_tr,fs,ch,1)

  Lab_ts = np.zeros(shape=(L_ts,1))
  for i in range(L_ts):
    Lab_ts[i] = int(sum(Y_test[i*fs:(i+1)*fs])/fs)

  Lab_ts = to_categorical(Lab_ts, num_classes=2)
  data_ts = X_test.reshape(L_ts,fs,ch,1)
  return data_tr, Lab_tr, data_ts, Lab_ts


# Y_hat: Predictions/Estimations made by the model
# Score: Probabilities of the classes
# Labels: Actual Labels in one-hot encoded form
def Performances(Labels, Score):

  Y_hat = np.round(Score)

  # Sensitivity/Recall,
  sen = metrics.Recall();
  sen.update_state(Labels, Y_hat)
  sen = sen.result()

  # Specificity/ True Negative Rate (TN/(TN+FP))
  fp = metrics.FalsePositives()
  fp.update_state(Labels, Y_hat)

  tn = metrics.TrueNegatives()
  tn.update_state(Labels, Y_hat)

  spe = tn.result()/(tn.result()+fp.result())

  # Accuracy
  acc = metrics.BinaryAccuracy();
  acc.update_state(Labels, Y_hat)
  acc = acc.result()

  # AUC_ROC
  auc = metrics.AUC(curve="ROC",
                    summation_method="interpolation");
  auc.update_state(Labels, Score)
  auc = auc.result()

  # performances
  perf = [sen, spe, acc, auc]

  return perf

In [ ]:
# Mounting Google Drive to get data
from google.colab import drive
drive.mount('/content/drive') #force_remount=False)
folder = '/content/drive/My Drive/Alcoholic_EEG_Research_Papers/Subjectwise_data/'
# ------------------------------------------------------------------------------
clear_output()

In [ ]:
# Dr Hamid's proposed model taken from kaggel
def create_seq_model(shape):
    model = Sequential()

    model.add(Conv1D(16, 15, input_shape = shape, activation='relu',
                            name="Conv1D_1"))
    model.add(MaxPooling1D(2, name="MaxPool_1"))
    model.add(BatchNormalization(name="Normalization_1"))
    model.add(Dropout(rate=0.4, name="Dropout_1"))

    model.add(Conv1D(32, 15, activation='relu', name="Conv1D_2"))
    model.add(MaxPooling1D(2, name="MaxPool_2"))
    model.add(BatchNormalization(name="Normalization_2"))
    model.add(Dropout(rate=0.4, name="Dropout_2"))

    model.add(Conv1D(64, 15,  activation='relu', name="Conv1D_3a"))
    model.add(Conv1D(64, 15,  activation='relu', name="Conv1D_3b"))

    model.add(GlobalMaxPool1D(name="MaxPool_3"))
    model.add(BatchNormalization(name="Normalization_3"))
    model.add(Dropout(rate=0.4, name="Dropout_3"))

    model.add(Dense(1, activation='sigmoid',
                    kernel_regularizer=l1_l2(l1=0.01, l2=0.01),
                    name="Dense_Final_Sigmoid", ))
    model.compile(optimizer=optimizers.RMSprop(0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [ ]:
# *** Main program to perform subjectwise 10-fold cross validation ***
K_test = 10;          # 10 fold inter-subject cross-validation
trails = int(16)      # 16 trails for each subject

Perf_seq_model = np.zeros((K_test,4),dtype=float)

seq_model_histories = []


cv = StratifiedKFold(n_splits=K_test,
                    shuffle=True,
                    random_state=66)
splits = cv.split(Sub,Y)

In [ ]:
i=0
for train,test in splits:
  trSub = Sub[train]
  tsSub = Sub[test]
  Y_train, X_train = Concatenate(trSub)
  Y_test, X_test = Concatenate(tsSub)
  win = int(len(Y_test)/(trails*len(tsSub)))
  scaler = StandardScaler()
  #X_train = scaler.fit_transform(X_train)
  #X_test = scaler.transform(X_test)
  trDim = X_train.shape
  X_train = X_train.reshape(trDim[0],trDim[1],1)
  tsDim = X_test.shape
  X_test = X_test.reshape(tsDim[0],tsDim[1],1)
  # Training
  data_tr, Lab_tr, data_ts, Lab_ts = Preprocessing_EEGNet(X_train,
                                                          Y_train,
                                                          X_test,
                                                          Y_test)

  seq_model = create_seq_model((256,16))
  Lab_tr = Lab_tr[:,0]
  Lab_ts = Lab_ts[:,0]
  seq_model_hist = seq_model.fit(data_tr,Lab_tr,
                             epochs = 100,
                             validation_split = 0.2,
                             shuffle=False,
                             callbacks = [early_stopping],
                             verbose = 0
  )
  seq_model_histories.append(seq_model_hist.history)

  # Testing
  sc_seq_model = seq_model.predict(data_ts)
  sc_seq_model = sc_seq_model.reshape(len(sc_seq_model),)
  # Performances

  Perf_seq_model[i,:] = Performances(Lab_ts, sc_seq_model)
  clear_output()
  print(Perf_seq_model[i,:])
  print("fold #: ", i)
  i = i+1

Perf_seq_model = pd.DataFrame(Perf_seq_model,
                          columns=['sensitivity','specificity','accuracy','AUCROC'])

In [ ]:
print('perf_seq_model','\n',
      round(Perf_seq_model.mean(),3), 'Parameters: ', seq_model.count_params())

perf_seq_model 
 sensitivity    0.397
specificity    0.550
accuracy       0.473
AUCROC         0.447
dtype: float64 Parameters:  104369


In [ ]:
import os
path = folder +"customAct/"

# Save the Performances
Perf_seq_model.to_csv(path + "Perf_seq_model.csv")
from tensorflow.keras.models import load_model

# Save the model
seq_model.save(path+"Models/seq_model.h5")

# Verify the model by printing its summary

with open(path+'Models/seq_model_histories.npy', 'wb') as f:
    np.save(f, seq_model_histories)
